# Week 9, Day 3: Fixed points and stability of nonlinear circuits

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w009/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
def S(x): return 1 / (1 + np.exp(-(x - 4)))
wEE, wEI, wIE, wII, hE, hI, tauI = 18.0, 10.0, 10.0, 2.0, 0.0, 0.0, 2.0
def F(z):
    E, I = z
    return np.array([-E + S(wEE * E - wEI * I + hE), (-I + S(wIE * E - wII * I + hI)) / tauI])
def jacobian(z, h=1e-6):
    J = np.zeros((2, 2))
    for j in range(2):
        e = np.zeros(2); e[j] = h
        J[:, j] = (F(z + e) - F(z - e)) / (2 * h)
    return J
print(F(np.array([0.5, 0.5])))

### Exercise: Find and classify every fixed point

Write `newton(z, steps=50)`, which repeats the Newton step using `jacobian`. Run it from a 6 by 6 grid of starts in the unit square, keeping each distinct answer (within 1e-6 of none found so far) where F is below 1e-10, in `fixed_points`. Then write `classify(z)` that returns 'stable', 'saddle' or 'unstable' from the eigenvalues of the Jacobian.

In [ ]:
def newton(z, steps=50):
    z = z.astype(float).copy()
    for _ in range(steps):
        ...
    return z

fixed_points = []
for E0 in np.linspace(0, 1, 6):
    for I0 in np.linspace(0, 1, 6):
        z = newton(np.array([E0, I0]))
        ...

def classify(z):
    re = np.linalg.eigvals(jacobian(z)).real
    ...

for z in sorted(fixed_points, key=lambda q: q[0]):
    print(z.round(4), classify(z), np.linalg.eigvals(jacobian(z)).round(3))

In [ ]:
import numpy as np
def test_newton():
    "newton converges to a point where F is zero"
    z = newton(np.array([0.9, 0.9]))
    assert np.linalg.norm(F(z)) < 1e-10

def test_count():
    "there are five distinct fixed points in the unit square"
    assert len(fixed_points) == 5
    for z in fixed_points:
        assert np.linalg.norm(F(z)) < 1e-9

def test_classes():
    "two are stable, two are saddles, and one is an unstable node"
    kinds = sorted(classify(z) for z in fixed_points)
    assert kinds.count("stable") == 2 and kinds.count("saddle") == 2 and kinds.count("unstable") == 1

test_newton()
test_count()
test_classes()
print("All checks passed")

### Exercise: Drive the inhibitory cells

Write `steady_rates(W, h)`. For a weakly coupled circuit ($w_{EE}$ = 0.5) and an inhibition-stabilised one ($w_{EE}$ = 2), compute the change in the inhibitory rate when the drive to the inhibitory cells rises by 0.5, storing the changes in `dI_weak` and `dI_isn`. Check the rates stay positive, so the threshold-linear neurons really are in their linear range.

In [ ]:
def J_lin(wEE, wEI=2.0, wIE=2.0, wII=0.5):
    return np.array([[wEE, -wEI], [wIE, -wII]])

def steady_rates(W, h):
    return ...

h0 = np.array([3.0, 1.0])
extra = np.array([0.0, 0.5])
dI_weak = ...
dI_isn = ...
for w in [0.5, 2.0]:
    print(f"w_EE = {w}: rates before {steady_rates(J_lin(w), h0).round(3)}, after {steady_rates(J_lin(w), h0 + extra).round(3)}")
print("change in inhibitory rate: weak", round(dI_weak, 3), " inhibition stabilised", round(dI_isn, 3))

In [ ]:
import numpy as np
def test_rates():
    "steady_rates solves r = W r + h"
    W = J_lin(1.0)
    r = steady_rates(W, np.array([3.0, 1.0]))
    assert np.allclose(r, W @ r + np.array([3.0, 1.0]))

def test_stable():
    "both circuits are stable: every eigenvalue of W - I has negative real part"
    for w in [0.5, 2.0]:
        assert np.all(np.linalg.eigvals(J_lin(w) - np.eye(2)).real < 0)

def test_paradox():
    "extra drive to I raises I's rate in the weak circuit and lowers it in the inhibition-stabilised one"
    assert dI_weak > 0 and dI_isn < 0

test_rates()
test_stable()
test_paradox()
print("All checks passed")